# Tutorial 5: Generative Inverse Design (Topology Optimization)

Usually, we ask: "What is the tortuosity of this material?"
Inverse design asks: "What material shape gives me the best tortuosity?"

In this tutorial, we will write a simple "Hill Climbing" optimization algorithm.
We will start with a random distribution of 50% solid and 50% pore. 
Then, we will iteratively swap solid and pore voxels. If the swap improves 
(lowers) the tortuosity, we keep it. If it gets worse, we revert it.

By strictly swapping voxels, we guarantee that the overall porosity stays 
exactly at 50%, forcing the algorithm to find the most efficient possible 
pathways for that specific volume fraction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import openimpala as oi

SIZE = 16 # Keep domain small so the tutorial runs in < 1 minute
ITERATIONS = 50

# 1. Initialize a completely random 50/50 microstructure
np.random.seed(42)
micro = np.random.choice([0, 1], size=(SIZE, SIZE, SIZE)).astype(np.int32)

In [ ]:
best_tau = np.inf
tau_history = []

print("Starting Topology Optimization...")

with oi.Session():
    # Evaluate initial state
    perc = oi.percolation_check(micro, phase=1, direction="z")
    if perc.percolates:
        best_tau = oi.tortuosity(micro, phase=1, direction="z").tortuosity
    
    print(f"Initial Tortuosity: {best_tau:.4f}")
    tau_history.append(best_tau)
    
    for i in range(ITERATIONS):
        # Find all current pore (1) and solid (0) coordinates
        pore_coords = np.argwhere(micro == 1)
        solid_coords = np.argwhere(micro == 0)
        
        # Pick one random pore and one random solid
        p_idx = pore_coords[np.random.choice(len(pore_coords))]
        s_idx = solid_coords[np.random.choice(len(solid_coords))]
        
        # Mutate: Swap them (this guarantees VF stays exactly the same)
        micro[tuple(p_idx)] = 0
        micro[tuple(s_idx)] = 1
        
        # Evaluate new physics
        perc = oi.percolation_check(micro, phase=1, direction="z")
        new_tau = np.inf
        if perc.percolates:
            new_tau = oi.tortuosity(micro, phase=1, direction="z").tortuosity
        
        # Hill Climbing logic: Only keep the swap if it improved transport!
        if new_tau < best_tau:
            best_tau = new_tau
            print(f"Iteration {i+1:02d} | SUCCESS | New Best Tau: {best_tau:.4f}")
        else:
            # Revert the mutation
            micro[tuple(p_idx)] = 1
            micro[tuple(s_idx)] = 0
            
        tau_history.append(best_tau)

In [ ]:
# Plot the optimization curve
plt.figure(figsize=(6, 4))
plt.plot(tau_history, 'g-', lw=2, marker='o')
plt.title("Topology Optimization (Fixed Porosity 50%)")
plt.xlabel("Mutation Iteration")
plt.ylabel("Tortuosity (Lower is better)")
plt.grid(True, alpha=0.3)
plt.show()

# Visualize the resulting "optimized" slice
plt.figure(figsize=(5, 5))
plt.imshow(micro[:, SIZE//2, :], cmap='gray')
plt.title("Optimized Microstructure Slice")
plt.axis('off')
plt.show()